<a href="https://colab.research.google.com/github/LiuChen-5749342/Generative-AI-and-AI-Applications/blob/main/Task_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Required Task 15**
(check the python notebook RAG_5_Multimodal_Chunking_OpenSourced.ipynb
for context and further details)

Fine-tune based on the data 'credit_card_aa.csv'. The data is also available at https://huggingface.co/datasets/priyaannamani/credit_card_qa

Create two fine-tuned models. The first one should take complaint as the input and predict policy_category. The second should take complaint as the input and predict resolution.

Take a sample of 60 records to fine-tune and evaluate the fine-tuned models on all the records in the dataset.

You can use either or both distilbert_base_uncased and distilgpt2 models. Performance comparative analysis of how well fine-tuning works.

Write a short narrative at the end to discuss your insights from fine-tuning.


# Install necessary libraries

In [1]:
!pip install transformers datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset, DatasetDict # Import DatasetDict here
import random
import torch

# Load the dataset 'credit_card_qa'


In [2]:
import pandas as pd

path = '/content/credit_card_qa.csv'
df = pd.read_csv(path)
df.head()

,complaint,relevant_policy,policy_category,resolution,validity
0,"Dana Wu, using card ending 0044, purchased $12...","You will earn 4% Cash Back on the first $25,00...",Cashback - 4%,Apply missing 4% cashback for the eligible gro...,Valid: Purchase was at an eligible merchant an...
1,Dana Wu is asking why she didn't get 4% cashba...,"You will earn 4% Cash Back on the first $25,00...",Cashback - 4%,Explain that wholesale clubs are often not cla...,Invalid: Merchant not classified as eligible g...
2,Dana Wu purchased $50 worth of electronics alo...,Some merchants may sell these products/service...,Cashback - 4%,Explain that only the grocery portion of the p...,Invalid: Non-food items purchased at a grocery...
3,Dana Wu's monthly electricity bill of $80 is a...,Recurring Bill Payments are payments made on a...,Cashback - 4%,Apply missing 4% cashback for the recurring el...,Valid: Eligible recurring utility bill payment...
4,Dana Wu's $300 car loan payment is automatical...,Recurring Bill Payments are payments made on a...,Cashback - 4%,Explain that loan payments are not typically c...,Invalid: Loan payments are not eligible recurr...


In [9]:
from sklearn.model_selection import train_test_split

# Keep only the columns needed
df = df[['complaint', 'policy_category', 'resolution']].dropna().reset_index(drop=True)

# Sample 60 records for fine-tuning
train_df = df.sample(n=60, random_state=42).reset_index(drop=True)

# Remaining records (optional holdout view)
remaining_df = df.drop(train_df.index, errors='ignore').reset_index(drop=True)

print("Full dataset shape:", df.shape)
print("Fine-tuning sample shape:", train_df.shape)
print("Remaining records shape:", remaining_df.shape)

train_df.head()

Full dataset shape: (80, 3)
Fine-tuning sample shape: (60, 3)
Remaining records shape: (20, 3)


,complaint,policy_category,resolution
0,Bob Kim paid his $99 annual fee for his card e...,Cashback - Exclusions,Explanation provided to user that annual fees ...
1,"Dana Wu, using card ending 0044, purchased $12...",Cashback - 4%,Apply missing 4% cashback for the eligible gro...
2,Bob Kim is frustrated because he didn't receiv...,Cashback - Exclusions,Explanation provided to user that cash advance...
3,Bob Kim used his card ending in 0022 to pay $1...,Cashback - Exclusions,Explanation provided to user that government p...
4,Dana Wu is complaining that her 4% cashback wa...,Cashback - 4%,Explain that cashback is earned on *net* purch...


In [10]:
from IPython.display import Markdown
display(Markdown(str(train_df[:5])))

                                           complaint        policy_category  \
0  Bob Kim paid his $99 annual fee for his card e...  Cashback - Exclusions   
1  Dana Wu, using card ending 0044, purchased $12...          Cashback - 4%   
2  Bob Kim is frustrated because he didn't receiv...  Cashback - Exclusions   
3  Bob Kim used his card ending in 0022 to pay $1...  Cashback - Exclusions   
4  Dana Wu is complaining that her 4% cashback wa...          Cashback - 4%   

                                          resolution  
0  Explanation provided to user that annual fees ...  
1  Apply missing 4% cashback for the eligible gro...  
2  Explanation provided to user that cash advance...  
3  Explanation provided to user that government p...  
4  Explain that cashback is earned on *net* purch...  

# Load the pre-trained language model and tokenizer

**For model 1:** It will take complaint as the input and predict policy_category.

In [14]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import Dataset

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Build label mapping for policy_category
policy_labels = sorted(df["policy_category"].unique())
id2label_policy = {i: label for i, label in enumerate(policy_labels)}
label2id_policy = {label: i for i, label in id2label_policy.items()}

# Encode labels in the 60-record fine-tuning set
train_df_model1 = train_df[["complaint", "policy_category"]].copy()
train_df_model1["label"] = train_df_model1["policy_category"].map(label2id_policy)

# Convert to Hugging Face Dataset
ds_train_model1 = Dataset.from_pandas(train_df_model1[["complaint", "label"]])

# Load model with correct number of labels
model_1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(policy_labels),
    id2label=id2label_policy,
    label2id=label2id_policy
)

def preprocess_function_model1(examples):
    tokenized_inputs = tokenizer(
        examples["complaint"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized_inputs["labels"] = examples["label"]
    return tokenized_inputs

tokenized_train_model1 = ds_train_model1.map(preprocess_function_model1, batched=True)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [15]:
# Print layer names
for name, param in model_1.named_parameters():
    print(name)

distilbert.embeddings.word_embeddings.weight
distilbert.embeddings.position_embeddings.weight
distilbert.embeddings.LayerNorm.weight
distilbert.embeddings.LayerNorm.bias
distilbert.transformer.layer.0.attention.q_lin.weight
distilbert.transformer.layer.0.attention.q_lin.bias
distilbert.transformer.layer.0.attention.k_lin.weight
distilbert.transformer.layer.0.attention.k_lin.bias
distilbert.transformer.layer.0.attention.v_lin.weight
distilbert.transformer.layer.0.attention.v_lin.bias
distilbert.transformer.layer.0.attention.out_lin.weight
distilbert.transformer.layer.0.attention.out_lin.bias
distilbert.transformer.layer.0.sa_layer_norm.weight
distilbert.transformer.layer.0.sa_layer_norm.bias
distilbert.transformer.layer.0.ffn.lin1.weight
distilbert.transformer.layer.0.ffn.lin1.bias
distilbert.transformer.layer.0.ffn.lin2.weight
distilbert.transformer.layer.0.ffn.lin2.bias
distilbert.transformer.layer.0.output_layer_norm.weight
distilbert.transformer.layer.0.output_layer_norm.bias
distil

In [16]:
# Calculate the total number of parameters
total_params = sum(p.numel() for p in model_1.parameters())

# Print the result
print(f"Total number of parameters: {total_params}")

Total number of parameters: 66969621


Fine Tune the model_1

In [20]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args_model1 = TrainingArguments(
    output_dir="./fine_tuned_policy_category_model",
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=5,
    report_to="none",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=10,
    save_strategy="epoch",
    eval_strategy="no"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_model1 = Trainer(
    model=model_1,
    args=training_args_model1,
    train_dataset=tokenized_train_model1,
    data_collator=data_collator
)

trainer_model1.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,2.273373
10,2.333429
15,2.026930
20,1.816300
25,2.005965
30,2.054039
35,1.688487
40,2.099002
45,1.592994
50,1.698798


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=75, training_loss=1.8463574600219728, metrics={'train_runtime': 187.879, 'train_samples_per_second': 1.597, 'train_steps_per_second': 0.399, 'total_flos': 9938421273600.0, 'train_loss': 1.8463574600219728, 'epoch': 5.0})

In [21]:
# Save the fine-tuned model_1 and tokenizer
save_path_model1 = "./saved_model1_policy_category"

trainer_model1.save_model(save_path_model1)
tokenizer.save_pretrained(save_path_model1)

print(f"Model 1 saved to: {save_path_model1}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model 1 saved to: ./saved_model1_policy_category


# Test the model_1

In [22]:
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Prepare the full dataset for evaluation
eval_df_model1 = df[["complaint", "policy_category"]].copy()
eval_df_model1["label"] = eval_df_model1["policy_category"].map(label2id_policy)

ds_eval_model1 = Dataset.from_pandas(eval_df_model1[["complaint", "label"]])

def preprocess_function_model1_eval(examples):
    tokenized_inputs = tokenizer(
        examples["complaint"],
        truncation=True,
        max_length=128
    )
    tokenized_inputs["labels"] = examples["label"]
    return tokenized_inputs

tokenized_eval_model1 = ds_eval_model1.map(preprocess_function_model1_eval, batched=True)

# Run predictions
pred_output_model1 = trainer_model1.predict(tokenized_eval_model1)

# Convert logits to predicted class IDs
y_pred_ids = np.argmax(pred_output_model1.predictions, axis=1)
y_true_ids = pred_output_model1.label_ids

# Convert IDs back to label names
y_pred_labels = [id2label_policy[i] for i in y_pred_ids]
y_true_labels = [id2label_policy[i] for i in y_true_ids]

# Metrics
acc_model1 = accuracy_score(y_true_labels, y_pred_labels)

print("Model 1 Accuracy on full dataset:", acc_model1)
print("\nClassification Report for Model 1:\n")
print(classification_report(y_true_labels, y_pred_labels))

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Model 1 Accuracy on full dataset: 0.625

Classification Report for Model 1:

                                           precision    recall  f1-score   support

                                 Cashback       0.00      0.00      0.00         1
                  Cashback - 1% Groceries       0.00      0.00      0.00         1
                            Cashback - 2%       0.00      0.00      0.00         3
                        Cashback - 2% Gas       0.00      0.00      0.00         2
                            Cashback - 4%       0.56      1.00      0.71        20
                   Cashback - Calculation       0.00      0.00      0.00         1
           Cashback - Eligible Categories       0.00      0.00      0.00         2
                    Cashback - Exclusions       0.33      1.00      0.50         7
                    Cashback - Expiration       0.00      0.00      0.00         1
          Cashback - Foreign Transactions       0.00      0.00      0.00         1
         

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Train the model_2

**For model 2:** It will take complaint as the input and predict resolution.

In [23]:
from transformers import AutoModelForSequenceClassification
from datasets import Dataset

# Build label mapping for resolution
resolution_labels = sorted(df["resolution"].unique())
id2label_resolution = {i: label for i, label in enumerate(resolution_labels)}
label2id_resolution = {label: i for i, label in id2label_resolution.items()}

# Encode labels in the 60-record fine-tuning set
train_df_model2 = train_df[["complaint", "resolution"]].copy()
train_df_model2["label"] = train_df_model2["resolution"].map(label2id_resolution)

# Convert to Hugging Face Dataset
ds_train_model2 = Dataset.from_pandas(train_df_model2[["complaint", "label"]])

# Load model_2
model_2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(resolution_labels),
    id2label=id2label_resolution,
    label2id=label2id_resolution
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
def preprocess_function_model2(examples):
    tokenized_inputs = tokenizer(
        examples["complaint"],
        truncation=True,
        max_length=128
    )
    tokenized_inputs["labels"] = examples["label"]
    return tokenized_inputs

tokenized_train_model2 = ds_train_model2.map(preprocess_function_model2, batched=True)

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [26]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args_model2 = TrainingArguments(
    output_dir="./fine_tuned_resolution_model",
    per_device_train_batch_size=4,
    num_train_epochs=5,
    logging_steps=5,
    report_to="none",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=10,
    save_strategy="epoch",
    eval_strategy="no"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_model2 = Trainer(
    model=model_2,
    args=training_args_model2,
    train_dataset=tokenized_train_model2,
    data_collator=data_collator
)

trainer_model2.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,4.383442
10,4.365079
15,4.390513
20,4.338713
25,4.330453
30,4.336247
35,4.319797
40,4.283694
45,4.256118
50,4.274651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=75, training_loss=4.280822168986003, metrics={'train_runtime': 133.7547, 'train_samples_per_second': 2.243, 'train_steps_per_second': 0.561, 'total_flos': 4725547188480.0, 'train_loss': 4.280822168986003, 'epoch': 5.0})

In [27]:
save_path_model2 = "./saved_model2_resolution"

trainer_model2.save_model(save_path_model2)
tokenizer.save_pretrained(save_path_model2)

print(f"Model 2 saved to: {save_path_model2}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model 2 saved to: ./saved_model2_resolution


In [28]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Prepare the full dataset for evaluation
eval_df_model2 = df[["complaint", "resolution"]].copy()
eval_df_model2["label"] = eval_df_model2["resolution"].map(label2id_resolution)

ds_eval_model2 = Dataset.from_pandas(eval_df_model2[["complaint", "label"]])

def preprocess_function_model2_eval(examples):
    tokenized_inputs = tokenizer(
        examples["complaint"],
        truncation=True,
        max_length=128
    )
    tokenized_inputs["labels"] = examples["label"]
    return tokenized_inputs

tokenized_eval_model2 = ds_eval_model2.map(preprocess_function_model2_eval, batched=True)

# Predict
pred_output_model2 = trainer_model2.predict(tokenized_eval_model2)

# Convert logits to predicted IDs
y_pred_ids_model2 = np.argmax(pred_output_model2.predictions, axis=1)
y_true_ids_model2 = pred_output_model2.label_ids

# Convert IDs back to label names
y_pred_labels_model2 = [id2label_resolution[i] for i in y_pred_ids_model2]
y_true_labels_model2 = [id2label_resolution[i] for i in y_true_ids_model2]

# Metrics
acc_model2 = accuracy_score(y_true_labels_model2, y_pred_labels_model2)

print("Model 2 Accuracy on full dataset:", acc_model2)
print("\nClassification Report for Model 2:\n")
print(classification_report(y_true_labels_model2, y_pred_labels_model2))

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Model 2 Accuracy on full dataset: 0.1375

Classification Report for Model 2:

                                                                                                                                                                                                                                                                                                                                    precision    recall  f1-score   support

                                                                                                                                                                      Apologize for the dissatisfaction and offer to have another concierge agent assist with more personalized recommendations. Review concierge service quality.       0.33      1.00      0.50         1
                                                                                                                                                                                                 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


The first model, which predicts policy_category from the complaint text, achieved an accuracy of 62.5% on the full dataset, suggesting that fine-tuning DistilBERT was reasonably effective for category classification, especially for broader and more repeated classes such as Purchase Security and Cashback - 4%. However, the classification report also shows that performance was uneven across labels. Many minority classes had precision, recall, and F1-scores of 0.00, indicating that the model largely failed to generalize to rare categories. This is consistent with the small fine-tuning sample of only 60 records and the highly imbalanced label distribution.

The second model, which predicts resolution from the complaint text, performed much worse, with an accuracy of only 13.75%. This result indicates that fine-tuning was not very successful for this task. The main reason is that resolution appears to be much more detailed and fine-grained than policy_category, with many unique or near-unique target responses. In effect, this makes the problem much harder, because the model is being asked to choose among many long, highly specific labels from very limited training data. DistilBERT sequence classification is generally better suited to a small set of repeated classes than to highly varied output texts.

The comparison shows that fine-tuning works better when the target labels are structured, limited, and repeated, as in the policy_category task. It works poorly when the target labels are long, numerous, and highly specific, as in the resolution task. A key insight from this experiment is that model performance depends not only on the model architecture, but also on how the prediction task is formulated. For the resolution task, a generative model such as DistilGPT2, or a reformulation into broader resolution groups, would likely be more appropriate than direct multi-class classification. Finally, since evaluation was conducted on the full dataset, including the 60 training examples, the reported scores may be somewhat optimistic and should not be interpreted as pure out-of-sample performance.